# Model benchmark: kit models × {per-class, multiclass} × split ratios × CV folds
- **Outer split** (by compound, stratified by class): 80/20, 75/25, 85/15 → held-out validation set.
- **Inner CV** on the training part: grouped, stratified k-fold, k = 4…10.
- **Validation**: fit on the training part, macro-F1 with a 1,000× bootstrap over validation compounds (95% CI).
- **Modes**: `per_class` = one binary model per class (one-vs-rest); `multiclass` = one model for all classes.
- **Biology**: every validation/test call is checked against the expected pathway readout of the called class (PROGENy, MSigDB Hallmark, HSF1 heat shock); robust z ≥ 1 = supported.
- Outputs go to `results/benchmark/`.

In [ ]:
import os, sys, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore"); os.environ["PYTHONWARNINGS"] = "ignore"

ROOT = Path(os.getenv("ROOT", ".."))   # notebook lives in src/
OUT = Path(os.getenv("OUT_DIR", ROOT / "results" / "benchmark")); (OUT / "submissions").mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "starter_kit")); os.environ["PYTHONPATH"] = str(ROOT / "starter_kit")  # workers need it too
from models import MODELS, make_model          # centroid, knn, logreg, rf, mlp
from evaluate import compound_metrics
from train import activity, aggregate

SEED, RATIOS, KS, MODES, N_BOOT, MAX_REPS, MIN_ACTIVE = 2026, [0.20, 0.25, 0.15], range(4, 11), ["per_class", "multiclass"], 1000, 6, 20
lab = lambda f: f"{round(100 * (1 - f))}/{round(100 * f)}"

## Data
The one compound with 300 replicate signatures (all at 20 µM) is capped at `MAX_REPS` signatures so it cannot dominate training.

In [ ]:
X = pd.read_csv(ROOT / "X_train.csv.gz", index_col=0)
y = pd.read_csv(ROOT / "y_train.csv", index_col=0)["moa"].loc[X.index]
meta = pd.read_csv(ROOT / "meta_train.csv", index_col=0).loc[X.index]
Xte = pd.read_csv(ROOT / "X_test.csv.gz", index_col=0); mte = pd.read_csv(ROOT / "meta_test.csv", index_col=0).loc[Xte.index]

ms = meta.sample(frac=1, random_state=SEED); keep = ms.index[ms.groupby("compound").cumcount() < MAX_REPS]
X, y, meta = X.loc[keep], y.loc[keep], meta.loc[keep]
cpd = y.groupby(meta.compound).first(); classes = sorted(cpd.unique())
allm = pd.concat([meta, mte]); assert allm.index.is_unique
print(X.shape, "train signatures |", len(cpd), "train compounds |", mte.compound.nunique(), "test compounds")

## Pathway readouts (biology check)
Compound profile = mean pathway z-score over doses ≥ 3.33 µM, scored against a gene-permutation null and robust-standardized over training compounds. `+` up, `-` down, `~` changed in either direction.

**Sources** (details, versions and rebuild script: `pathway_resources/README.md`): PROGENy footprints — Schubert et al. 2018, [doi:10.1038/s41467-017-02391-6](https://doi.org/10.1038/s41467-017-02391-6), Bioconductor `progeny` 1.32.0; MSigDB Hallmark — Liberzon et al. 2015, [doi:10.1016/j.cels.2015.12.004](https://doi.org/10.1016/j.cels.2015.12.004), release 2026.1 via `msigdbr` 26.1.1 ([Zenodo 18968178](https://zenodo.org/records/18968178)); probe→symbol `hgu133a.db` 3.13.0; HSF1 heat-shock set hand-picked. The class → readout mapping (`PRIORS`) is our own, from target biology.

In [ ]:
W = pd.read_csv(ROOT / "pathway_resources" / "l1000_pathway_weights.csv", index_col=0).drop(columns="symbol")

def pathway_z(Xs, n_perm=500, rng=np.random.default_rng(SEED)):
    Xv, Wv = Xs[W.index].values, W.values
    null = np.stack([Xv @ Wv[rng.permutation(len(Wv))] for _ in range(n_perm)])
    return pd.DataFrame((Xv @ Wv - null.mean(0)) / null.std(0), index=Xs.index, columns=W.columns)

hi = allm.dose_um >= 3.33
P = pathway_z(pd.concat([X, Xte])[hi]).groupby(allm.compound[hi]).mean()
med = P.loc[cpd.index].median(); P = (P - med) / (1.4826 * (P.loc[cpd.index] - med).abs().median())

PRIORS = {"EGFR": "PROGENy:EGFR- PROGENy:MAPK-", "MEK": "PROGENy:MAPK- PROGENy:EGFR-", "RAF": "PROGENy:MAPK-",
          "PI3K": "PROGENy:PI3K- H:MTORC1_SIGNALING-", "AKT": "PROGENy:PI3K- H:MTORC1_SIGNALING-", "mTOR": "H:MTORC1_SIGNALING- PROGENy:PI3K-",
          "JAK": "PROGENy:JAK-STAT- H:IL6_JAK_STAT3_SIGNALING-", "CDK": "H:E2F_TARGETS- H:G2M_CHECKPOINT-",
          "PLK": "H:G2M_CHECKPOINT~ H:MITOTIC_SPINDLE~", "tubulin": "H:G2M_CHECKPOINT~ H:MITOTIC_SPINDLE~",
          "topoisomerase": "PROGENy:p53+ H:E2F_TARGETS-", "PARP": "PROGENy:p53+ H:DNA_REPAIR~",
          "HSP": "custom:HSF1_heat_shock+", "proteasome": "H:UNFOLDED_PROTEIN_RESPONSE+ custom:HSF1_heat_shock+", "HDAC": ""}
PRIORS = {next(c for c in classes if c.startswith(k + " ")): [(t[:-1], {"+": 1, "-": -1, "~": 0}[t[-1]]) for t in v.split()] for k, v in PRIORS.items()}

def support(compounds, calls):
    """Mean signed readout of the called class per compound (NaN = class has no readout, e.g. HDAC)."""
    return pd.Series([np.mean([s * P.loc[c, f] if s else abs(P.loc[c, f]) for f, s in PRIORS[k]]) if PRIORS[k] else np.nan
                      for c, k in zip(compounds, calls)], index=list(compounds))

## Models, modes, splits

In [ ]:
def fit_predict(name, Xf, yf, Xp):
    keep = activity(Xf) >= MIN_ACTIVE                       # kit's activity filter (training rows only)
    if yf[keep].nunique() < yf.nunique(): keep[:] = True
    m = make_model(name)
    if name == "rf": m.est.set_params(n_jobs=1)             # parallelism is across jobs instead
    try: m.fit(Xf[keep], yf[keep])
    except ValueError:                                     # MLP early stopping on tiny classes (as in train.py)
        if name != "mlp": raise
        m.est.set_params(mlpclassifier__early_stopping=False); m.fit(Xf[keep], yf[keep])
    return m.predict_proba(Xp)

def scores(name, mode, Xf, yf, Xp):
    """Compound × class scores; doses combined with the kit's activity-weighted mean."""
    if mode == "multiclass": S = fit_predict(name, Xf, yf, Xp)
    else: S = pd.DataFrame({c: fit_predict(name, Xf, yf.where(yf == c, "rest"), Xp)[c] for c in sorted(yf.unique())})
    return aggregate(S.reindex(columns=classes, fill_value=0.0), allm.compound, activity(Xp))

def holdout(frac, rng=None):
    """Per class, `frac` of its compounds (≥1) go to validation."""
    rng = np.random.default_rng(SEED)
    return [c for _, g in cpd.groupby(cpd) for c in rng.permutation(g.index)[:max(1, round(frac * len(g)))]]

def run(frac, name, mode, k):
    tr = ~meta.compound.isin(holdout(frac)); Xt, yt = X[tr], y[tr]
    if k:   # inner CV on the training part
        cv = StratifiedGroupKFold(k, shuffle=True, random_state=SEED).split(Xt, yt, meta.compound[tr])
        S = pd.concat([scores(name, mode, Xt.iloc[a], yt.iloc[a], Xt.iloc[b]) for a, b in cv])
        return "cv", dict(ratio=lab(frac), model=name, mode=mode, k=k, **compound_metrics(S, cpd.loc[S.index]))
    return "val", (lab(frac), name, mode, scores(name, mode, Xt, yt, X[~tr]))

print({lab(f): len(holdout(f)) for f in RATIOS}, "validation compounds")

## Run everything (parallel)

In [ ]:
jobs = [(f, n, m, k) for n in ["rf", "mlp", "logreg", "knn", "centroid"] for m in MODES for f in RATIOS for k in [*KS, 0]]
out = Parallel(n_jobs=-1)(delayed(run)(*j) for j in jobs)
test = dict(zip([(n, m) for n in MODELS for m in MODES],
                Parallel(n_jobs=-1)(delayed(scores)(n, m, X, y, Xte) for n in MODELS for m in MODES)))
cv = pd.DataFrame([r for t, r in out if t == "cv"]); cv.to_csv(OUT / "inner_cv.csv", index=False)
cv.pivot_table(index=["model", "mode"], columns="k", values="macro_f1").round(2)  # mean over the 3 ratios

## Validation: bootstrap + pathway check

In [ ]:
rng = np.random.default_rng(SEED); rows, vp, pc = [], [], []
for r, n, m, S in [v for t, v in out if t == "val"]:
    t, p = cpd.loc[S.index], S.idxmax(1); sup = support(S.index, p); ok, s = t == p, sup >= 1
    f1 = lambda i: f1_score(t.values[i], p.values[i], average="macro", labels=np.unique(t.values[i]), zero_division=0)
    b = [f1(i) for i in rng.integers(0, len(t), (N_BOOT, len(t)))]
    rows.append(dict(ratio=r, model=n, mode=m, n_val=len(t), macro_f1=f1(slice(None)), boot_median=np.median(b),
                     ci_lo=np.percentile(b, 2.5), ci_hi=np.percentile(b, 97.5), frac_supported=s.mean(),
                     acc_supported=ok[s].mean(), acc_unsupported=ok[~s & sup.notna()].mean()))
    vp.append(pd.DataFrame(dict(ratio=r, model=n, mode=m, true=t, pred=p, readout=sup)).rename_axis("compound"))
    pc.append(pd.Series(f1_score(t, p, labels=classes, average=None, zero_division=0), index=classes, name=(r, n, m)))
val = pd.DataFrame(rows).round(3); val.to_csv(OUT / "validation_bootstrap.csv", index=False)
pd.concat(vp).to_csv(OUT / "validation_predictions.csv")
pd.DataFrame(pc).round(2).rename_axis(["ratio", "model", "mode"]).to_csv(OUT / "validation_per_class_f1.csv")
val.sort_values("macro_f1", ascending=False).head(10)

## Test predictions: one submission per model × mode, with pathway support

In [ ]:
for (n, m), S in test.items(): S.rename_axis("compound").reset_index().to_csv(OUT / "submissions" / f"submission_{n}_{m}.csv", index=False)
calls = pd.DataFrame({f"{n}|{m}": S.idxmax(1) for (n, m), S in test.items()})
tsup = pd.DataFrame({c: support(calls.index, calls[c]) for c in calls})
calls.join(tsup.round(2), rsuffix=" readout").to_csv(OUT / "test_calls_with_pathway_readout.csv")
tsum = pd.DataFrame({"frac_supported": (tsup >= 1).mean(), "n_no_readout": tsup.isna().sum(),
                     "agree_with_consensus": calls.eq(calls.mode(axis=1)[0], axis=0).mean()}).round(2)
tsum.to_csv(OUT / "test_pathway_summary.csv"); tsum.sort_values("frac_supported", ascending=False)

## Figures

In [ ]:
cols = dict(zip(MODELS, plt.cm.tab10.colors)); ls = {"multiclass": "-", "per_class": "--"}
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, r in zip(axs, map(lab, RATIOS)):
    for (n, m), g in cv[cv.ratio == r].groupby(["model", "mode"]): ax.plot(g.k, g.macro_f1, ls[m], color=cols[n], marker="o", ms=3, label=f"{n} ({m})")
    ax.set(title=f"inner CV, split {r}", xlabel="CV folds (k)")
axs[0].set_ylabel("compound macro-F1"); axs[-1].legend(fontsize=7, frameon=False, bbox_to_anchor=(1, 1)); fig.tight_layout(); fig.savefig(OUT / "fig_inner_cv_vs_k.png", dpi=150)

fig, axs = plt.subplots(1, 3, figsize=(13, 3.8), sharex=True, sharey=True)
order = [(n, m) for n in MODELS for m in MODES]
for ax, r in zip(axs, map(lab, RATIOS)):
    v = val[val.ratio == r].set_index(["model", "mode"]).loc[order]
    for i, ((n, m), row) in enumerate(v.iterrows()):
        ax.plot([row.ci_lo, row.ci_hi], [i, i], color=cols[n], ls=ls[m]); ax.plot(row.macro_f1, i, "o", color=cols[n], mfc="white" if m == "per_class" else cols[n])
    ax.set(title=f"validation {r} (n={int(v.n_val.iloc[0])}), 95% bootstrap CI", xlabel="macro-F1", yticks=range(len(order)), yticklabels=[f"{n} | {m}" for n, m in order])
fig.tight_layout(); fig.savefig(OUT / "fig_validation_bootstrap.png", dpi=150)

g = val.groupby(["model", "mode"])[["acc_supported", "acc_unsupported", "frac_supported"]].mean().loc[order].join(tsum.set_axis(pd.MultiIndex.from_tuples([tuple(i.split("|")) for i in tsum.index])), rsuffix="_test")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True); yy = np.arange(len(order))
axs[0].scatter(g.acc_supported, yy, color="#3b6ea8", label="pathway-supported calls"); axs[0].scatter(g.acc_unsupported, yy, color="#c9773a", label="unsupported calls")
axs[0].set(xlabel="validation accuracy (mean over splits)", yticks=yy, yticklabels=[f"{n} | {m}" for n, m in order], title="Is a pathway-supported call more often right?"); axs[0].legend(fontsize=7, frameon=False)
axs[1].barh(yy, g.frac_supported_test, color="#7a7a7a"); axs[1].set(xlabel="fraction of test calls pathway-supported", title="Test predictions: biological support", xlim=(0, 1))
fig.tight_layout(); fig.savefig(OUT / "fig_pathway_check.png", dpi=150)